In [1]:

# install java and pyspark
!pip uninstall -y pyspark dataproc-spark-connect 2>/dev/null
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.0 --quiet

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

print('Setup complete.')

Found existing installation: pyspark 4.0.2
Uninstalling pyspark-4.0.2:
  Successfully uninstalled pyspark-4.0.2
Found existing installation: dataproc-spark-connect 1.1.0
Uninstalling dataproc-spark-connect-1.1.0:
  Successfully uninstalled dataproc-spark-connect-1.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.5 MB/s eta 0:00:00
Setup complete.


In [7]:
from pyspark.sql import SparkSession
# starting the spark session
spark = SparkSession.builder.appName('Ecom').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)
from google.colab import files
uploaded = files.upload() #uploaded my kaggle.json


Spark version: 3.5.0


Saving kaggle.json to kaggle.json


In [8]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
!pip install kaggle --quiet
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store # download the ZIP the dataset

print('Download done.')

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors
100% 4.29G/4.29G [03:19<00:00, 23.0MB/s]

Download done.


In [9]:
!unzip -o ecommerce-behavior-data-from-multi-category-store.zip -d /content/ecom_raw
!mkdir -p /content/ecom_stream
!head -n 30001 /content/ecom_raw/2019-Oct.csv > /content/ecom_stream/sample.csv
!wc -l /content/ecom_stream/sample.csv
print('Sample ready.')

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: /content/ecom_raw/2019-Nov.csv  
  inflating: /content/ecom_raw/2019-Oct.csv  
30001 /content/ecom_stream/sample.csv
Sample ready.


In [10]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import (
    session_window, col, count, sum as spark_sum, when, to_timestamp, regexp_replace
)
event_schema = StructType([
    StructField('event_time',    StringType(), True),
    StructField('event_type',    StringType(), True),
    StructField('product_id',    StringType(), True),
    StructField('category_id',   StringType(), True),
    StructField('category_code', StringType(), True),
    StructField('brand',         StringType(), True),
    StructField('price',         DoubleType(), True),
    StructField('user_id',       StringType(), True),
    StructField('user_session',  StringType(), True),
])
ecom_stream = (
    spark.readStream
    .schema(event_schema)
    .option('header', True)
    .option('maxFilesPerTrigger', 1)
    .csv('/content/ecom_stream')
)


clean_stream = ecom_stream.withColumn(
    'event_time',
    to_timestamp(regexp_replace(col('event_time'), ' UTC', ''), 'yyyy-MM-dd HH:mm:ss')
)

#(30 min gap)
sessions = (
    clean_stream
    .withWatermark('event_time', '10 minutes')
    .groupBy(
        col('user_id'),
        session_window(col('event_time'), '30 minutes')
    )
    .agg(
        spark_sum(when(col('event_type') == 'cart', 1).otherwise(0)).alias('cart_count'),
        spark_sum(when(col('event_type') == 'purchase', 1).otherwise(0)).alias('purchase_count'),
        count('*').alias('n_events')
    )
)

print('Pipeline ready.')

Pipeline ready.


In [17]:
from pyspark.sql.functions import lit
import time

# fraud rule: more than 5 cart adds and no purchase in the session
alerts = (
    sessions
    .filter((col('cart_count') > 5) & (col('purchase_count') == 0))
    .select(
        col('user_id'),
        col('session_window.start').alias('session_start'),
        col('cart_count'),
        col('purchase_count'),
        lit('FRAUD').alias('status')
    )
)

def show_batch(batch_df, batch_id):
    print('Batch:', batch_id)
    batch_df.show(truncate=False)

query = (
    alerts
    .writeStream
    .foreachBatch(show_batch)
    .outputMode('complete')
    .start()
)

time.sleep(20)
query.stop()

Batch: 0
+---------+-------------------+----------+--------------+------+
|user_id  |session_start      |cart_count|purchase_count|status|
+---------+-------------------+----------+--------------+------+
|513053986|2019-10-01 02:40:35|6         |0             |FRAUD |
+---------+-------------------+----------+--------------+------+

